In [ ]:
import pandas as pd

df = pd.read_csv("combined_scaled_battery_data.csv")

print(df.shape)
print(df.columns)
print(df.head())
print(df.describe())

(15064, 9)
Index(['Cycle_Index', 'Discharge Time (s)', 'Decrement 3.6-3.4V (s)',
       'Max. Voltage Dischar. (V)', 'Min. Voltage Charg. (V)',
       'Time at 4.15V (s)', 'Time constant current (s)', 'Charging time (s)',
       'RUL'],
      dtype='object')
   Cycle_Index  Discharge Time (s)  Decrement 3.6-3.4V (s)  \
0          1.0             2595.30             1151.488500   
1          2.0             7408.64             1172.512500   
2          3.0             7393.76             1112.992000   
3          4.0             7385.50             1080.320667   
4          6.0            65022.75            29813.487000   

   Max. Voltage Dischar. (V)  Min. Voltage Charg. (V)  Time at 4.15V (s)  \
0                      3.670                    3.211           5460.001   
1                      4.246                    3.220           5508.992   
2                      4.249                    3.224           5508.993   
3                      4.250                    3.225           

In [2]:
!pip install xgboost
!pip install scikit-learn

## XGBoost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

In [ ]:
NUM_TEST_BATTERIES = 8

FEATURE_COLS = [
    "Discharge Time (s)",
    "Decrement 3.6-3.4V (s)",
    "Max. Voltage Dischar. (V)",
    "Min. Voltage Charg. (V)",
    "Time at 4.15V (s)",
    "Time constant current (s)",
    "Charging time (s)",
]

df = pd.read_csv("combined_scaled_battery_data.csv")
print(f"Loaded dataset: {df.shape}")


def split_into_battery_segments(dataframe: pd.DataFrame) -> list:
    """Split dataframe into per-battery segments by detecting RUL resets."""
    rul_values = dataframe["RUL"].values
    rul_diffs = np.diff(rul_values)
    boundary_indices = np.where(rul_diffs > 0)[0] + 1

    segment_starts = np.concatenate([[0], boundary_indices])
    segment_ends   = np.concatenate([boundary_indices, [len(dataframe)]])

    return [
        dataframe.iloc[start:end].reset_index(drop=True)
        for start, end in zip(segment_starts, segment_ends)
    ]


battery_segments = split_into_battery_segments(df)
print(f"Total batteries: {len(battery_segments)}")

calce_count = sum(1 for s in battery_segments if s["Is_NASA"].iloc[0] == 0)
nasa_count  = sum(1 for s in battery_segments if s["Is_NASA"].iloc[0] == 1)
print(f"  HNEI/CALCE batteries: {calce_count}")
print(f"  NASA batteries:       {nasa_count}")

In [ ]:
# Battery-held-out split: same stratified sampling as CNN
rng = np.random.default_rng(42)
calce_indices = [i for i, s in enumerate(battery_segments) if s["Is_NASA"].iloc[0] == 0]
nasa_indices  = [i for i, s in enumerate(battery_segments) if s["Is_NASA"].iloc[0] == 1]

num_test_calce = NUM_TEST_BATTERIES // 2
num_test_nasa  = NUM_TEST_BATTERIES - num_test_calce
test_indices = (
    list(rng.choice(calce_indices, size=num_test_calce, replace=False)) +
    list(rng.choice(nasa_indices,  size=num_test_nasa,  replace=False))
)
test_index_set = set(test_indices)

train_segments = [s for i, s in enumerate(battery_segments) if i not in test_index_set]
test_segments  = [battery_segments[i] for i in test_indices]
print(f"Batteries — train: {len(train_segments)}, test: {len(test_segments)}")

In [ ]:
# XGBoost uses flat per-cycle features (no sliding window needed)
def segments_to_arrays(segments: list) -> tuple:
    all_features = pd.concat(segments)[FEATURE_COLS].values.astype(np.float32)
    all_labels   = pd.concat(segments)["RUL"].values.astype(np.float32)
    return all_features, all_labels


X_train, y_train = segments_to_arrays(train_segments)
X_test,  y_test  = segments_to_arrays(test_segments)
print(f"Train rows: {X_train.shape}, Test rows: {X_test.shape}")

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

xgb_model.fit(X_train, y_train)
print("Training complete.")

In [ ]:
y_pred = xgb_model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"XGBoost (battery-held-out)")
print(f"  MAE:  {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  R²:   {r2:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.3, s=10)
min_val, max_val = y_test.min(), y_test.max()
ax.plot([min_val, max_val], [min_val, max_val], linestyle="--", color="red")
ax.set_xlabel("True RUL")
ax.set_ylabel("Predicted RUL")
ax.set_title("XGBoost: Predicted vs True RUL (battery-held-out)")
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
importances = xgb_model.feature_importances_

feat_imp_df = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Importance": importances
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(8, 4))
plt.bar(feat_imp_df["Feature"], feat_imp_df["Importance"])
plt.xticks(rotation=45, ha="right")
plt.title("XGBoost Feature Importance")
plt.tight_layout()
plt.show()